In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import os
import mediapipe as mp
import torch
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
# ==========================================\n# 1. SETUP & KONFIGURASI
# ==========================================
print("Loading RAFT model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# Gunakan weights bawaan PyTorch
weights = Raft_Large_Weights.DEFAULT
transforms = weights.transforms()

# Load model dan pindahkan ke GPU/CPU
raft_model = raft_large(weights=weights, progress=False).to(device)
raft_model.eval() # Set ke evaluation mode
print("RAFT model loaded!")

TARGET_ALIGN_SIZE = (320, 320)
LEFT_EYE_IDX = [33, 133, 159, 145]
RIGHT_EYE_IDX = [362, 263, 386, 374]
EAR_LEFT_EYE = [33, 160, 158, 133, 153, 144]
EAR_RIGHT_EYE = [362, 385, 387, 263, 373, 380]
BLINK_THRESHOLD = 0.22

ROI_INDICES = {
    "area_dahi": [109, 104, 333, 338],
    "area_alis_kanan": [104, 46, 55, 107],
    "area_alis_kiri": [333, 276, 285, 336],
    "area_mata_kanan": [46, 111, 114, 55],
    "area_mata_kiri": [276, 340, 343, 285],
    "area_antara_alis": [151, 55, 168, 285],
    "area_pipi_kanan": [114, 117, 216, 98],
    "area_pipi_kiri": [343, 346, 436, 294],
    "area_hidung": [114, 164, 343, 168],
    "area_mulut_kanan": [216, 169, 200, 0],
    "area_mulut_kiri": [436, 394, 200, 0],

    "area_pangkal_hidung": [168, 193, 195, 417],
}

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True, max_num_faces=1, refine_landmarks=True, min_detection_confidence=0.5
)

# ==========================================\n# 2. HELPER FUNCTIONS ALIGNMENT & EAR
# ==========================================
def detect_face(image_rgb):
    results = face_mesh.process(image_rgb)
    if not results.multi_face_landmarks: return None
    face_landmarks = results.multi_face_landmarks[0]
    h, w, _ = image_rgb.shape
    points = np.array([(lm.x * w, lm.y * h) for lm in face_landmarks.landmark])
    return (int(np.min(points[:, 0])), int(np.min(points[:, 1])), 
            int(np.max(points[:, 0]) - np.min(points[:, 0])), 
            int(np.max(points[:, 1]) - np.min(points[:, 1])))

def detect_eye_positions_and_blink(image_rgb, landmarks):
    h, w, _ = image_rgb.shape
    
    # 1. Gunakan HANYA sudut mata (canthus) yang stabil untuk alignment
    # Kiri: 33 (outer), 133 (inner) | Kanan: 362 (inner), 263 (outer)
    left_eye = np.mean([[landmarks.landmark[i].x * w, landmarks.landmark[i].y * h] for i in [33, 133]], axis=0)
    right_eye = np.mean([[landmarks.landmark[i].x * w, landmarks.landmark[i].y * h] for i in [362, 263]], axis=0)
    
    # 2. Hitung EAR untuk deteksi kedipan langsung
    ear_left = compute_ear(landmarks, EAR_LEFT_EYE)
    ear_right = compute_ear(landmarks, EAR_RIGHT_EYE)
    is_blinking = (ear_left < BLINK_THRESHOLD) or (ear_right < BLINK_THRESHOLD)
    
    return left_eye, right_eye, is_blinking

def compute_ear(landmarks, eye_indices):
    p1 = np.array([landmarks.landmark[eye_indices[0]].x, landmarks.landmark[eye_indices[0]].y])
    p2 = np.array([landmarks.landmark[eye_indices[1]].x, landmarks.landmark[eye_indices[1]].y])
    p3 = np.array([landmarks.landmark[eye_indices[2]].x, landmarks.landmark[eye_indices[2]].y])
    p4 = np.array([landmarks.landmark[eye_indices[3]].x, landmarks.landmark[eye_indices[3]].y])
    p5 = np.array([landmarks.landmark[eye_indices[4]].x, landmarks.landmark[eye_indices[4]].y])
    p6 = np.array([landmarks.landmark[eye_indices[5]].x, landmarks.landmark[eye_indices[5]].y])
    d1 = np.linalg.norm(p2 - p6)
    d2 = np.linalg.norm(p3 - p5)
    d3 = np.linalg.norm(p1 - p4)
    return (d1 + d2) / (2.0 * d3)

def get_square_box(box, img_shape, margin=0.15):
    h_img, w_img = img_shape[:2]
    x, y, w, h = box
    center_x, center_y = x + w // 2, y + h // 2
    max_dim = max(w, h)
    side_length = int(max_dim * (1 + margin * 2))
    new_x = max(0, center_x - side_length // 2)
    new_y = max(0, center_y - side_length // 2)
    if new_x + side_length > w_img: new_x = max(0, w_img - side_length)
    if new_y + side_length > h_img: new_y = max(0, h_img - side_length)
    final_w = min(side_length, w_img - new_x)
    final_h = min(side_length, h_img - new_y)
    return new_x, new_y, final_w, final_h

def align_and_crop_face(bgr_img, target_size=(320, 320), enable_align=True, smooth_state=None, alpha=0.15, alpha_size=0.02):
    if bgr_img is None: return None, smooth_state
    
    if smooth_state is None: 
        smooth_state = {'angle': 0.0, 'center': None, 'box': None, 'is_blinking': False}
        
    rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    
    results = face_mesh.process(rgb_img)
    if not results.multi_face_landmarks: return None, smooth_state
    landmarks = results.multi_face_landmarks[0]
    
    if enable_align:
        # Asumsi fungsi ini mengembalikan (left_eye_pos, right_eye_pos, is_blinking)
        left_eye, right_eye, is_blinking = detect_eye_positions_and_blink(rgb_img, landmarks)
        smooth_state['is_blinking'] = is_blinking 
        
        dx, dy = right_eye[0] - left_eye[0], right_eye[1] - left_eye[1]
        raw_angle = np.degrees(np.arctan2(dy, dx))
        raw_center = ((left_eye[0] + right_eye[0]) / 2.0, (left_eye[1] + right_eye[1]) / 2.0)
        
        if smooth_state['center'] is None: 
            smooth_state['angle'], smooth_state['center'] = raw_angle, raw_center
        else:
            # PERBAIKAN 1: Hapus 'if not is_blinking'. Tetap lakukan smoothing agar tidak loncat saat berkedip.
            smooth_state['angle'] = (alpha * raw_angle) + ((1 - alpha) * smooth_state['angle'])
            smooth_state['center'] = (
                (alpha * raw_center[0]) + ((1 - alpha) * smooth_state['center'][0]),
                (alpha * raw_center[1]) + ((1 - alpha) * smooth_state['center'][1])
            )
            
        h, w = bgr_img.shape[:2]
        M = cv2.getRotationMatrix2D(smooth_state['center'], smooth_state['angle'], 1.0)
        processed_bgr = cv2.warpAffine(bgr_img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    else:
        processed_bgr = bgr_img.copy()
        smooth_state['is_blinking'] = False

    # Ekstrak face_box dari gambar hasil rotasi
    rotated_results = face_mesh.process(cv2.cvtColor(processed_bgr, cv2.COLOR_BGR2RGB))
    if rotated_results.multi_face_landmarks:
        rot_lms = rotated_results.multi_face_landmarks[0]
        h_rot, w_rot = processed_bgr.shape[:2]
        rot_points = np.array([(lm.x * w_rot, lm.y * h_rot) for lm in rot_lms.landmark])
        
        # Biarkan raw bounding box dalam bentuk desimal / float
        min_x, min_y = np.min(rot_points[:, 0]), np.min(rot_points[:, 1])
        max_x, max_y = np.max(rot_points[:, 0]), np.max(rot_points[:, 1])
        face_box = (min_x, min_y, max_x - min_x, max_y - min_y)
    else:
        return None, smooth_state
    
    if smooth_state['box'] is None:
        smooth_state['box'] = face_box
    else:
        x, y, w_box, h_box = face_box
        px, py, pw, ph = smooth_state['box']
        
        # 1. DECOUPLE UKURAN (W, H)
        new_w = (alpha_size * w_box) + ((1 - alpha_size) * pw)
        new_h = (alpha_size * h_box) + ((1 - alpha_size) * ph)
        
        # 2. DECOUPLE POSISI 
        curr_cx = x + (w_box / 2.0)
        curr_cy = y + (h_box / 2.0)
        
        prev_cx = px + (pw / 2.0)
        prev_cy = py + (ph / 2.0)
        
        new_cx = (alpha * curr_cx) + ((1 - alpha) * prev_cx)
        new_cy = (alpha * curr_cy) + ((1 - alpha) * prev_cy)
        
        new_x = new_cx - (new_w / 2.0)
        new_y = new_cy - (new_h / 2.0)
        
        # PERBAIKAN 2: Simpan dalam bentuk FLOAT di smooth_state. Jangan di-convert ke int!
        smooth_state['box'] = (new_x, new_y, new_w, new_h)
    
    # PERBAIKAN 3: Convert ke integer (membulatkan) hanya pada saat akan dikirim ke fungsi crop
    box_for_crop = tuple(int(round(val)) for val in smooth_state['box'])
    
    sq_x, sq_y, sq_w, sq_h = get_square_box(box_for_crop, processed_bgr.shape, margin=0.15)
    cropped = processed_bgr[sq_y:sq_y + sq_h, sq_x:sq_x + sq_w]
    
    if cropped.size == 0: return None, smooth_state
    
    return cv2.resize(cropped, target_size, interpolation=cv2.INTER_AREA), smooth_state
# ==========================================\n# 3. OPTICAL FLOW & ROI ACCUMULATION
# ==========================================
def compute_optical_flow(prev_bgr, curr_bgr, model=raft_model, device=device, transforms=transforms):
    """
    Menghitung optical flow menggunakan algoritma RAFT (PyTorch).
    """
    prev_rgb = cv2.cvtColor(prev_bgr, cv2.COLOR_BGR2RGB)
    curr_rgb = cv2.cvtColor(curr_bgr, cv2.COLOR_BGR2RGB)
    
    h_asli, w_asli = prev_rgb.shape[:2]

    img1_tensor = torch.from_numpy(prev_rgb).permute(2, 0, 1).unsqueeze(0)
    img2_tensor = torch.from_numpy(curr_rgb).permute(2, 0, 1).unsqueeze(0)

    img1_batch, img2_batch = transforms(img1_tensor, img2_tensor)
    
    img1_batch = img1_batch.to(device)
    img2_batch = img2_batch.to(device)

    with torch.no_grad(): 
        list_of_flows = model(img1_batch, img2_batch, num_flow_updates=24)
        predicted_flow = list_of_flows[-1] 

    flow_numpy = predicted_flow[0].permute(1, 2, 0).cpu().numpy()
    
    flow_numpy = flow_numpy[:h_asli, :w_asli]
    
    return flow_numpy

def get_roi_dominant_flow(flow, landmarks, img_w, img_h):
    """Mengekstrak arah gerakan (u, v) dominan untuk setiap ROI tanpa saling meniadakan"""
    roi_flows = {}
    
    # Fungsi pembantu untuk mencari gerakan paling ekstrem (mengabaikan pembatalan)
    def get_dominant_movement(matrix):
        if matrix.size == 0: return 0.0
        
        # Ambil nilai di persentil ke-95 (tarikan positif terkuat) 
        # dan persentil ke-5 (tarikan negatif terkuat).
        # Menggunakan persentil lebih aman dari np.max/np.min agar tidak menangkap noise.
        p95 = np.percentile(matrix, 95)
        p5 = np.percentile(matrix, 5)
        
        # Pilih gerakan yang tarikannya paling kuat/jauh dari nol
        if abs(p95) > abs(p5):
            return p95
        else:
            return p5

    for name, indices in ROI_INDICES.items():
        pts = np.array([[int(landmarks.landmark[i].x * img_w), 
                         int(landmarks.landmark[i].y * img_h)] for i in indices])
        
        x_min, y_min = np.min(pts, axis=0)
        x_max, y_max = np.max(pts, axis=0)
        
        if name == "area_dahi":
            roi_h = y_max - y_min
            y_min = max(0, int(y_min - roi_h * 0.6))
            
        x_min, y_min = max(0, x_min), max(0, y_min)
        x_max, y_max = min(img_w, x_max), min(img_h, y_max)
        
        crop_flow = flow[y_min:y_max, x_min:x_max]
        
        if crop_flow.size == 0:
            roi_flows[name] = (0.0, 0.0)
        else:
            # Cari tarikan terkuat di sumbu X (u) dan sumbu Y (v)
            du = get_dominant_movement(crop_flow[..., 0])
            dv = get_dominant_movement(crop_flow[..., 1])
            roi_flows[name] = (du, dv)
    
    return roi_flows

# ==========================================\n# 4. ENERGY CALCULATION & DETECTION
# ==========================================
def calculate_accumulated_flow_energy(frames_bgr, drift_decay=0.92, motion_threshold=0.5, blink_pad=3):
    """
    Menghitung energi berdasarkan AKUMULASI posisi arah.
    """
    
    # === PASS 1: PRE-SCAN (Mencari Landmark & Memperlebar Kedipan) ===
    cached_landmarks = []
    raw_blinks = []
    
    for curr_bgr in frames_bgr:
        curr_rgb = cv2.cvtColor(curr_bgr, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(curr_rgb)
        
        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0]
            cached_landmarks.append(landmarks)
            
            ear_left = compute_ear(landmarks, EAR_LEFT_EYE)
            ear_right = compute_ear(landmarks, EAR_RIGHT_EYE)
            raw_blinks.append((ear_left < BLINK_THRESHOLD) or (ear_right < BLINK_THRESHOLD))
        else:
            cached_landmarks.append(None)
            raw_blinks.append(False)
            
    expanded_blinks = [False] * len(raw_blinks)
    for i, is_blinking in enumerate(raw_blinks):
        if is_blinking:
            start_idx = max(0, i - blink_pad)
            end_idx = min(len(raw_blinks), i + blink_pad + 1)
            for j in range(start_idx, end_idx):
                expanded_blinks[j] = True

    # === PASS 2: OPTICAL FLOW & AKUMULASI DECAY DINAMIS ===
    total_energy = [0]
    cached_flows = []
    global_compensations = []  # Menyimpan (global_du, global_dv) per frame untuk visualisasi
    accum_flow = {name: np.array([0.0, 0.0]) for name in ROI_INDICES.keys()}
    stationary_counter = {name: 1 for name in ROI_INDICES.keys()}
    
 
    history_per_roi = {name: [0.0] for name in ROI_INDICES.keys() if name != "area_pangkal_hidung"}

    for i in range(len(frames_bgr) - 1):
        prev_bgr = frames_bgr[i]
        curr_bgr = frames_bgr[i+1]
        
        flow = compute_optical_flow(prev_bgr, curr_bgr, model=raft_model)
        cached_flows.append(flow)

        landmarks = cached_landmarks[i]
        is_blinking_expanded = expanded_blinks[i+1]
        
        instant_energy = 0
        if landmarks:
            h, w = curr_bgr.shape[:2]
            roi_flows = get_roi_dominant_flow(flow, landmarks, w, h)
            global_du, global_dv = roi_flows.get("area_pangkal_hidung", (0.0, 0.0))
            global_compensations.append((global_du, global_dv))

            roi_magnitudes = []
            for name, (du, dv) in roi_flows.items():
                if name == "area_pangkal_hidung":
                    continue

                du_bersih = du - global_du
                dv_bersih = dv - global_dv

                if name in ["area_mata_kanan", "area_mata_kiri"]:
                    du_bersih, dv_bersih = 0.0, 0.0

                current_movement = np.hypot(du_bersih, dv_bersih)
                
                if current_movement > motion_threshold:
                    current_decay = 1.0 
                    stationary_counter[name] = 1
                else:
                    current_decay = drift_decay ** stationary_counter[name]
                    stationary_counter[name] = min(15, stationary_counter[name] + 1)

                accum_flow[name][0] = (accum_flow[name][0] * current_decay) + du_bersih
                accum_flow[name][1] = (accum_flow[name][1] * current_decay) + dv_bersih
                
                mag = np.linalg.norm(accum_flow[name])
                roi_magnitudes.append(mag)
                
               
                history_per_roi[name].append(mag)

            instant_energy = np.sum(roi_magnitudes)
           
        else:
            global_compensations.append((0.0, 0.0))
            for name in history_per_roi:
                history_per_roi[name].append(0.0)
        
        total_energy.append(instant_energy)
        
    return np.array(total_energy), cached_flows, history_per_roi, global_compensations

def detect_multiple_expressions_raw(magnitudes, fps=30, min_prominence=0.6, min_height=0.8):
    """
    min_prominence: Seberapa menonjol minimal sebuah puncak dari kakinya.
    min_height: Ketinggian absolut minimal agar tidak menangkap noise di ROI datar.
    """
    raw_mag = np.array(magnitudes)
    
    # 1. Gunakan Robust Statistics (Persentil) daripada Mean/Std
    # Jika ROI dominan diam, percentil 95 akan tetap merepresentasikan puncak sebenarnya (jika ada)
    p95 = np.percentile(raw_mag, 95)
    median_mag = np.median(raw_mag)
    
    # 2. Tentukan Prominence dan Height yang lebih cerdas
    optimal_prominence = max((p95 - median_mag) * 0.5, min_prominence)
    optimal_height = max(p95 * 0.5, min_height)

    # 3. Syarat Lebar (Width)
    # Micro-expression biasanya berdurasi > 100ms (sekitar 3-4 frame di 30fps).
    # Lebar diukur pada setengah tinggi prominence (rel height = 0.5)
    min_width = max(int(fps * 0.1), 3) 

    peaks, properties = find_peaks(
        raw_mag, 
        distance=max(int(fps/5), 6),
        prominence=optimal_prominence,
        height=optimal_height,
        width=min_width 
    )

    # Baseline threshold untuk memotong onset/offset
    sorted_mag = np.sort(raw_mag)
    baseline_threshold = np.mean(sorted_mag[:int(max(1, len(raw_mag)*0.3))])
    
    d_mag = np.gradient(raw_mag)
    grad_std = np.std(d_mag)
    
    # 4. Batas bawah gradien agar pencarian onset/offset tidak nyangkut di noise
    grad_thresh = max(0.5 * grad_std, 0.05) 
    
    patience_limit = max(3, int(fps * 0.1))
    results = []

    for apex in peaks:
        onset, offset = 0, len(raw_mag) - 1
        
        # --- MENCARI ONSET (Mundur dari Apex) ---
        patience_count = 0
        for i in range(apex, 0, -1):
            if d_mag[i] <= grad_thresh: 
                patience_count += 1
            else: 
                patience_count = 0
                
            if patience_count >= patience_limit or raw_mag[i] <= baseline_threshold:
                onset = min(apex, i + patience_count) 
                break

        # --- MENCARI OFFSET (Maju dari Apex) ---
        patience_count = 0
        for i in range(apex, len(raw_mag) - 1):
            if d_mag[i] >= -grad_thresh: 
                patience_count += 1
            else: 
                patience_count = 0
                
            if patience_count >= patience_limit or raw_mag[i] <= baseline_threshold:
                offset = max(apex, i - patience_count)
                break

        results.append({
            "onset": onset, 
            "apex": apex, 
            "offset": offset,
            "prominence": properties['prominences'][list(peaks).index(apex)] if 'prominences' in properties else 0
        })

    return results, raw_mag

def merge_roi_events(history_per_roi, fps=30):
    all_events = []
    
    # 1. Deteksi event pada tiap ROI secara mandiri
    for roi_name, history in history_per_roi.items():
        # Lewati ROI yang nyaris tidak ada pergerakan (flat/kosong)
        if np.max(history) < 0.1:
            continue
            
        events, _ = detect_multiple_expressions_raw(magnitudes=history, fps=fps)
        
        # Tandai event ini milik ROI mana
        for e in events:
            e['roi'] = roi_name 
            all_events.append(e)
            
    if not all_events:
        return []
        
    # 2. Urutkan semua event berdasarkan waktu Onset (kiri ke kanan)
    all_events = sorted(all_events, key=lambda x: x['onset'])
    
    # 3. Proses Penggabungan (Merging) yang Overlap
    merged_events = []
    current_event = all_events[0].copy()
    
    for next_event in all_events[1:]:
        # Cek Overlap: Apakah event berikutnya mulai sebelum event saat ini selesai?
        # (Kita berikan toleransi gap 1 frame agar gerakan yang sangat berdekatan ikut tergabung)
        if next_event['onset'] <= current_event['offset'] + 1:
            
            # Rentangkan batas waktu gabungan
            current_event['onset'] = min(current_event['onset'], next_event['onset'])
            current_event['offset'] = max(current_event['offset'], next_event['offset'])
            
            # Bandingkan nilai absolut pergerakan di titik Apex masing-masing ROI
            mag_current = history_per_roi[current_event['roi']][current_event['apex']]
            mag_next = history_per_roi[next_event['roi']][next_event['apex']]
            
            # Jika Apex event berikutnya lebih kuat, jadikan dia sebagai Apex Utama
            if mag_next > mag_current:
                current_event['apex'] = next_event['apex']
                current_event['roi'] = next_event['roi'] # Simpan nama ROI pemenang
                current_event['prominence'] = next_event['prominence']
                
        else:
            # Jika tidak overlap, simpan event sebelumnya ke daftar hasil
            merged_events.append(current_event)
            # Mulai melacak event baru
            current_event = next_event.copy()
            
    # Jangan lupa masukkan event terakhir
    merged_events.append(current_event)
    
    return merged_events

# ==========================================\n# 5. VISUALIZATION HELPER
# ==========================================
def flow_to_color(flow):
    hsv = np.zeros((flow.shape[0], flow.shape[1], 3), dtype=np.uint8)
    hsv[..., 1] = 255
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
# ==========================================
# 6. MAIN EXECUTION
# ==========================================
input_dir = "input"
frame_dir = "sp15j"

full_path = os.path.join(input_dir, frame_dir)

if not os.path.exists(full_path):
    print("Folder not found.")
    exit()

frame_files = sorted(
    [f for f in os.listdir(full_path) if f.endswith((".jpg", ".png"))],
    key=lambda x: int(''.join(filter(str.isdigit, x)) or 0)
)
raw_frames = [cv2.imread(os.path.join(full_path, f)) for f in frame_files]
frames_aligned = []
current_smooth_state = None 

valid_frame_numbers = []
valid_file_names = [] 

print("Aligning frames...")
for i, img in enumerate(raw_frames):
    aligned, current_smooth_state = align_and_crop_face(img, TARGET_ALIGN_SIZE, enable_align=True, smooth_state=current_smooth_state)
    if aligned is not None: 
        frames_aligned.append(aligned)
        file_num = int(''.join(filter(str.isdigit, frame_files[i])) or 0)
        valid_frame_numbers.append(file_num)
        valid_file_names.append(frame_files[i])

print("Calculating Accumulated Flow Energy...")
# ---> MENERIMA 3 OUTPUT SEKARANG <---
raw_magnitudes, cached_flows, history_per_roi, global_compensations = calculate_accumulated_flow_energy(frames_aligned, drift_decay=0.96, motion_threshold=0.1) 
raw_mags = np.array([np.max([history_per_roi[roi][i] for roi in history_per_roi]) for i in range(len(raw_magnitudes))])

# Deteksi menggunakan metode FUSION (Merge ROI)
detected_events = merge_roi_events(history_per_roi, fps=30)

print(f"Total Frames: {len(raw_mags)}")
print(f"Detected Expressions: {len(detected_events)}")

# ==========================================
# PRE-FORMATTING TEKS HASIL DETEKSI
# ==========================================
bgr_colors = [(0, 0, 255), (0, 165, 255), (128, 0, 128), (42, 42, 165)] 
event_texts = []

max_idx = len(valid_file_names) - 1
for idx, event in enumerate(detected_events):
    onset_idx = min(event['onset'] + 1, max_idx)
    apex_idx = min(event['apex'] + 1, max_idx)
    offset_idx = min(event['offset'] + 1, max_idx)
    
    onset_name = valid_file_names[onset_idx]
    apex_name = valid_file_names[apex_idx]
    offset_name = valid_file_names[offset_idx]
    # Ambil nama ROI penyumbang Apex terbesar agar tampil di video
    main_muscle = event['roi'].replace("area_", "").replace("_", " ").title()
    
    line = f"[{idx+1}] {onset_name} -> {apex_name} -> {offset_name} | Apex: {main_muscle} (Prom: {event['prominence']:.1f})"

    event_texts.append({
        'lines': [line],
        'color': bgr_colors[idx % len(bgr_colors)],
        'onset': event['onset'],
        'offset': event['offset']
    })

# ==========================================
# 7. VIDEO GENERATOR (FULL HD 1920x1080)
# ==========================================
print("\nGenerating Video Layout...")
CANVAS_W, CANVAS_H = 1920, 1080
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)  # bikin folder kalau belum ada

output_filename = os.path.join(output_dir, f'{frame_dir}_test_raft3_flicker.mp4')

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_filename, fourcc, 30.0, (CANVAS_W, CANVAS_H))

colors = ['red', 'orange', 'purple', 'brown']

# --- SETUP GRAFIK GLOBAL (Sisi Kanan Atas) ---
fig_g, ax_g = plt.subplots(figsize=(8.0, 2.6), dpi=100)
ax_g.plot(raw_mags, label='Global Energy', linewidth=1.5, color='blue')
for idx, event in enumerate(detected_events):
    c = colors[idx % len(colors)]
    ax_g.axvspan(event['onset'], event['offset'], color=c, alpha=0.15)
    ax_g.axvline(event['apex'], color=c, linestyle='-', linewidth=1)
ax_g.set_title("Global Accumulated Flow Energy", fontsize=11, fontweight='bold')
ax_g.set_xlim(0, len(raw_mags))
ax_g.set_ylim(0, max(raw_mags) * 1.2 if max(raw_mags) > 0 else 1.0) 
vline_g = ax_g.axvline(x=0, color='black', linewidth=1.5, linestyle='--')
fig_g.tight_layout(pad=1.0)

# --- SETUP GRAFIK GRID ROI (Bagian Bawah) ---
valid_rois = list(history_per_roi.keys())
fig_r, axes_r = plt.subplots(3, 4, figsize=(19.2, 7.6), dpi=100)
axes_r = axes_r.flatten()
vlines_r = []

for idx, roi_name in enumerate(valid_rois):
    ax = axes_r[idx]
    history = history_per_roi[roi_name]
    ax.plot(history, color='green', linewidth=1.2)
    
    for e_idx, event in enumerate(detected_events):
        c = colors[e_idx % len(colors)]
        ax.axvspan(event['onset'], event['offset'], color=c, alpha=0.15)
        
    ax.set_title(roi_name.replace("area_", "").replace("_", " ").title(), fontsize=10)
    ax.set_xlim(0, len(history))
    ax.set_ylim(0, max(history) * 1.2 if max(history) > 0 else 1.0)
    vl = ax.axvline(x=0, color='black', linewidth=1, linestyle='--')
    vlines_r.append(vl)

# Sembunyikan subplot yang kosong (jika ada)
for idx in range(len(valid_rois), len(axes_r)):
    axes_r[idx].axis('off')

fig_r.tight_layout(pad=1.5)
print("Rendering Video Frames...")
MAX_VISIBLE_ITEMS = 6

# Tentukan lebar frame yang ingin ditampilkan di grafik pada satu waktu
# Misalnya 150 frame (kira-kira 5 detik jika 30 FPS)
WINDOW_SIZE = 150 
total_frames = len(raw_mags)

for i in range(len(frames_aligned) - 1):
    curr_bgr = frames_aligned[i+1]
    current_file_num = valid_frame_numbers[i+1]
    
    # Bersihkan flow dari gerakan global (pangkal hidung) sebelum divisualisasikan
    raw_flow = cached_flows[i]
    g_du, g_dv = global_compensations[i]
    cleaned_flow = raw_flow.copy()
    cleaned_flow[..., 0] -= g_du  # Kurangi komponen horizontal global
    cleaned_flow[..., 1] -= g_dv  # Kurangi komponen vertikal global
    flow_bgr = flow_to_color(cleaned_flow)
    
    # ==========================================
    # LOGIKA AUTO-SCROLL GRAFIK (SLIDING WINDOW)
    # ==========================================
    # Kita buat agar garis saat ini (i) selalu berada di tengah grafik
    x_min = max(0, i - (WINDOW_SIZE // 2))
    x_max = x_min + WINDOW_SIZE
    
    # Jika sudah mentok di ujung kanan, tahan batas x_min agar lebar grafik stabil
    if x_max > total_frames:
        x_max = max(WINDOW_SIZE, total_frames)
        x_min = max(0, x_max - WINDOW_SIZE)

    # 1. Update x-axis untuk grafik Global
    ax_g.set_xlim(x_min, x_max)
    vline_g.set_xdata([i, i])
    
    # 2. Update x-axis untuk seluruh grafik Grid ROI
    for idx in range(len(valid_rois)):
        axes_r[idx].set_xlim(x_min, x_max)
        
    for vl in vlines_r:
        vl.set_xdata([i, i])
    # ==========================================
        
    # Render Matplotlib Global ke numpy array
    fig_g.canvas.draw()
    graph_g_img = np.frombuffer(fig_g.canvas.buffer_rgba(), dtype=np.uint8)
    graph_g_img = graph_g_img.reshape(fig_g.canvas.get_width_height()[::-1] + (4,))
    graph_g_bgr = cv2.cvtColor(graph_g_img, cv2.COLOR_RGBA2BGR)
    
    # Render Matplotlib Grid ROI ke numpy array
    fig_r.canvas.draw()
    graph_r_img = np.frombuffer(fig_r.canvas.buffer_rgba(), dtype=np.uint8)
    graph_r_img = graph_r_img.reshape(fig_r.canvas.get_width_height()[::-1] + (4,))
    graph_r_bgr = cv2.cvtColor(graph_r_img, cv2.COLOR_RGBA2BGR)
    
    # === KOMPOSISI KANVAS ===
    canvas = np.zeros((CANVAS_H, CANVAS_W, 3), dtype=np.uint8)
    canvas[:] = (25, 25, 25) # Warna background gelap elegan
    
    # 1. Tempel Wajah & Flow (Kiri Atas)
    canvas[38:38+320, 20:20+320] = curr_bgr
    canvas[38:38+320, 264:264+320] = flow_bgr
    cv2.putText(canvas, f"File Frame: {current_file_num}", (20, 25), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(canvas, "Optical Flow", (264, 25), cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    
    # 2. Tempel Grafik Global (Tengah Atas)
    canvas[20:20+260, 510:510+800] = graph_g_bgr
    
    # 3. Tempel Grid ROI (Bagian Bawah)
    canvas[320:1080, 0:1920] = graph_r_bgr
    
    # 4. Tulis Teks Auto-Scroll (Kanan Atas)
    text_x = 1330
    text_y = 45
    cv2.putText(canvas, f"Detected ME ({len(event_texts)}):", (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX, 0.65, (255, 255, 255), 1)
    text_y += 35
    
    current_active_idx = 0
    for idx, e in enumerate(detected_events):
        if i >= e['onset']: current_active_idx = idx
            
    start_idx = max(0, current_active_idx - 2)
    end_idx = min(len(event_texts), start_idx + MAX_VISIBLE_ITEMS)
    if end_idx - start_idx < MAX_VISIBLE_ITEMS and len(event_texts) > MAX_VISIBLE_ITEMS:
        start_idx = len(event_texts) - MAX_VISIBLE_ITEMS

    if start_idx > 0:
        cv2.putText(canvas, "  . . . (scroll) . . .", (text_x + 80, text_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)

    for idx in range(start_idx, end_idx):
        e_info = event_texts[idx]
        is_active = (e_info['onset'] <= i <= e_info['offset'])
        font_scale = 0.55 if is_active else 0.45
        thickness = 2 if is_active else 1
        color = e_info['color'] if is_active else (150, 150, 150) 
        
        for line in e_info['lines']:
            cv2.putText(canvas, line, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, thickness)
            text_y += 22
        text_y += 10 
        
    if end_idx < len(event_texts):
        cv2.putText(canvas, "  . . . (scroll) . . .", (text_x + 80, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)
    
    out.write(canvas)

out.release()
plt.close(fig_g)
plt.close(fig_r)
print(f"Selesai! Video HD disimpan sebagai '{output_filename}'")

Loading RAFT model...
Menggunakan device: cuda
RAFT model loaded!
Aligning frames...
Calculating Accumulated Flow Energy...
Total Frames: 1569
Detected Expressions: 10

Generating Video Layout...
Rendering Video Frames...
Selesai! Video HD disimpan sebagai 'output\sp15j_test_raft3_flicker.mp4'
